# 提示词工程：释放大模型潜力的关键艺术

## 一、提示词工程的核心价值

在大型语言模型应用开发中，提示词(Prompt)设计是连接人类意图与AI能力的桥梁。精心设计的提示词如同精准的编程指令，能够引导模型生成符合预期的响应，而粗糙的提示则可能导致结果偏离目标。提示词工程的价值主要体现在：

1. **精准表达需求**：通过结构化描述确保模型准确理解任务要求
2. **控制输出质量**：设定明确的格式、风格和内容规范
3. **提升交互效率**：减少反复调试和结果修正的成本
4. **解锁高级功能**：实现复杂的信息处理和创造性工作

## 基础环境配置

首先配置基础环境和DeepSeek客户端：

In [ ]:
from dotenv import load_dotenv, find_dotenv
from langchain_deepseek import ChatDeepSeek

def load_deepseek_config():
    """安全加载DeepSeek API配置"""
    # 自动查找并加载.env文件
    load_dotenv(find_dotenv())

load_deepseek_config()

llm = ChatDeepSeek(
    model="deepseek-chat",
    max_retries=2
)

def get_completion(text):
    messages = [
        ("system", "你是Ai助手，可以帮助我回答生活中的各种问题"),
        ("human", text),
    ]
    response = llm.invoke(messages)
    return response.content


def get_completion_from_messages(messages, temperature=0):
    """
        messages —— 消息列表，对话历史
        temperature —— 控制生成文本的随机性 / 创造性，浮点数(0 ~ 1)
    """
    llm.temperature = temperature
    response = llm.invoke(messages)
    return response.content

## 二、实用提示词模式解析

### 1. 基础问答模式

最简单的提示词形式，直接提出问题。适用于：
- 事实性查询
- 定义解释
- 简单计算
- 常识问答

代码示例：

In [ ]:
prompt = "英国的首都是哪里？"
print(get_completion(prompt))

### 2. 模板化输出模式

通过预定义输出结构，确保结果格式统一。适用于：
- 数据提取
- 报告生成
- 信息卡片制作
- 结构化知识呈现

代码示例：

In [ ]:
prompt = """请用以下格式回答关于城市的问题：
城市名称：<名称>
所属国家：<国家>
著名景点：<景点1>, <景点2>, <景点3>

问题：请介绍英国的首都"""
print(get_completion(prompt))

### 3. 参数化动态模式

使用变量实现提示词复用。适用于：
- 批量处理相似任务
- 多轮对话保持一致性
- 个性化内容生成
- 条件分支场景

代码示例：

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template_string = """
城市名称：<名称>
所属国家：<国家>
著名景点：<景点1>, <景点2>, <景点3>

问题：请介绍{country}的首都
"""
template = ChatPromptTemplate.from_template(template_string)
country = "法国"
messages = template.format_messages(country=country)
print(get_completion_from_messages(messages=messages, temperature=0.3))

利用提示词技巧，可以让模型产生结构化输出。如下述方法，可以帮我产生10个NBA球星的具体数据，并以 JSON 的格式进行返回

In [ ]:
def json_generate(names, fields, num, des):
    num = 10 if num > 10 else num
    namePart = "、".join(names)
    fieldsPart = "、".join(fields)
    template_string = """
    我想要获取{des}相关内容的数据，\
    请生成包含{namePart}的{num}条数据，\
    并以 Json 格式提供，其中包含以下键：{fieldsPart}。\
    """
    template = ChatPromptTemplate.from_template(template_string)
    messages = template.format_messages(des=des, namePart=namePart, num=num, fieldsPart=fieldsPart)
    return get_completion_from_messages(messages=messages)


names = ["姓名", "年龄", "生涯总得分", "生涯总助攻", "生涯总篮板"]
fields = ["name", "age", "scores", "assistant", "rebound"]
num = 10
des = "NBA篮球明星"

print(json_generate(names, fields, num, des))

利用提示词让模型按照 Markdown 的形式输出一个完成具体目标的步骤

In [ ]:
def detail_step(target):
    template_string = """
    我需要知道{target}，请分步骤说明，每个步骤包含：\
    - 步骤编号\
    - 步骤名称\
    - 详细说明\
    - 注意事项\

    用Markdown格式输出
    """
    template = ChatPromptTemplate.from_template(template_string)
    messages = template.format_messages(target=target)
    return get_completion_from_messages(messages=messages)

print(detail_step("如何泡一杯完美的绿茶？"))

## 四、高级应用场景

通过提示词实现：
- 文本摘要生成
- 情感倾向分析
- 关键词提取
- 数据格式转换

示例应用：
1. 自动提取文档的关键词

In [ ]:
def extractor(text, num_words=3):
    template_string = """
    您的任务是根据一段描述来从中提取关键词。\
    请对三个反引号之间的文本进行提取关键词操作，请帮我提取{num_words}个关键词。\
    文本内容: ```{text}```\
     """
    template = ChatPromptTemplate.from_template(template_string)
    messages = template.format_messages(text=text, num_words=num_words)
    return get_completion_from_messages(messages=messages)

text = """
一场3：3让国足迎来大喜讯，伊万又有新发现，锋线喜添天才神童 \n
北京时间3月28日晚，中超第3轮，上海海港客场挑战实力一般的青岛西海岸。赛前，大家都认为上海海港可以轻松取胜，然而，比赛却最终以一场令人意想不到的3:3 进球大战收场。这场比赛不仅充满了戏剧性，更让一位年仅19岁的本土小将在中超赛场上横空出世，一战成名。\n
他就是今年刚刚被提拔进上海海港一线队的天才小将李新翔。此役，李新翔迎来了他职业生涯极为重要的中超首秀。原本人们以为他只是短暂亮相，走一下过场而已，却没想到，他却出道即巅峰，打出了一鸣惊人的高光表现。此役之前，没有人知道他；此役过后，没有人不知道他。\n
中超首秀，大部分小将会心烦意乱，发挥失常，但李新翔却心理素质强大，一夜之间从青铜变成为赛场上的王者。比赛开场之后，客场作战的上海海港队状态不佳，显得十分慢热，而主队青岛西海岸却气势如虹，不到20分钟便连进两球，海港队0:2 落后，局势岌岌可危。\n
就在这时，李新翔挺身而出，临危救主。第25分钟，他在左路得球后，凭借出色的个人能力快速摆脱对方后卫，紧接着拔脚怒射，皮球如炮弹般直飞球门，干净利落地洞穿了青岛西海岸的球门。这一进球过程一气呵成，充分展现了李新翔卓越的射门天赋，让在场的观众和电视机前的球迷们击节赞叹。\n
下半场易地再战，李新翔的表现依旧是球队进攻端最具威胁的一环。第55分钟，他在禁区左路送出一脚极具想象力的过顶传球，皮球在空中划出一道美妙的弧线，精准地落在了古斯塔沃身前，后者轻松头球破门。第83分钟，李新翔再次在门前展现出了他出色的阅读比赛能力和传球脚法，助攻帮助莱昂纳多在混战中完成破门。最终，海港队3:3艰难逼平青岛西海岸，避免了一场失利。而在这场比赛中，19岁的李新翔凭借一个进球和两个助攻，一人独造3球，成为了全场最闪耀的人。\n
小将李新翔的出色表现对中国国家队来说无疑是一个重大喜讯。目前，国家队在人才储备方面正迫切需要年轻且有实力的新星加入。国足主帅伊万在看到李新翔这场精彩绝伦的比赛之后，很难不被他的天赋和潜力所吸引。鉴于 6 月份即将到来的世预赛，伊万极有可能对李新翔破格提拔，让他有机会在更高级别的赛事中为国家队效力。这位锋线天才神童的出现，无疑为国足注入了新的活力与希望，相信他能在6月的世预赛上真正一战成名，一鸣惊人，带给国足两连胜。\n
"""

print(extractor(text, 5))

2. 自动推断文本情感倾向，并用一个词加以表述

In [ ]:
def recognizer(text):
    template_string = """
    您的任务是根据一段描述来从中推断作者的情感倾向。\
    请对三个反引号之间的文本进行情感倾向的推断操作，并使用一个词加以描述。\
    文本内容: ```{text}```\
     """
    template = ChatPromptTemplate.from_template(template_string)
    messages = template.format_messages(text=text)
    return get_completion_from_messages(messages=messages)

text = """
日常价9.82，双十二18.8，如果不是偶然发现，妥妥冤大头
"""

print(recognizer(text))

3. 简要概括一段文本的内容，并生成摘要

In [ ]:
def summarizer(text, num_words=100):
    template_string = """
    您的任务是根据一段描述来生成简短摘要。\
    请对三个反引号之间的文本进行概括，最多{num_words}个字。\
    文本内容: ```{text}```\
     """
    template = ChatPromptTemplate.from_template(template_string)
    messages = template.format_messages(text=text, num_words=num_words)
    return get_completion_from_messages(messages=messages)

text = """
一场3：3让国足迎来大喜讯，伊万又有新发现，锋线喜添天才神童 \n
北京时间3月28日晚，中超第3轮，上海海港客场挑战实力一般的青岛西海岸。赛前，大家都认为上海海港可以轻松取胜，然而，比赛却最终以一场令人意想不到的3:3 进球大战收场。这场比赛不仅充满了戏剧性，更让一位年仅19岁的本土小将在中超赛场上横空出世，一战成名。\n
他就是今年刚刚被提拔进上海海港一线队的天才小将李新翔。此役，李新翔迎来了他职业生涯极为重要的中超首秀。原本人们以为他只是短暂亮相，走一下过场而已，却没想到，他却出道即巅峰，打出了一鸣惊人的高光表现。此役之前，没有人知道他；此役过后，没有人不知道他。\n
中超首秀，大部分小将会心烦意乱，发挥失常，但李新翔却心理素质强大，一夜之间从青铜变成为赛场上的王者。比赛开场之后，客场作战的上海海港队状态不佳，显得十分慢热，而主队青岛西海岸却气势如虹，不到20分钟便连进两球，海港队0:2 落后，局势岌岌可危。\n
就在这时，李新翔挺身而出，临危救主。第25分钟，他在左路得球后，凭借出色的个人能力快速摆脱对方后卫，紧接着拔脚怒射，皮球如炮弹般直飞球门，干净利落地洞穿了青岛西海岸的球门。这一进球过程一气呵成，充分展现了李新翔卓越的射门天赋，让在场的观众和电视机前的球迷们击节赞叹。\n
下半场易地再战，李新翔的表现依旧是球队进攻端最具威胁的一环。第55分钟，他在禁区左路送出一脚极具想象力的过顶传球，皮球在空中划出一道美妙的弧线，精准地落在了古斯塔沃身前，后者轻松头球破门。第83分钟，李新翔再次在门前展现出了他出色的阅读比赛能力和传球脚法，助攻帮助莱昂纳多在混战中完成破门。最终，海港队3:3艰难逼平青岛西海岸，避免了一场失利。而在这场比赛中，19岁的李新翔凭借一个进球和两个助攻，一人独造3球，成为了全场最闪耀的人。\n
小将李新翔的出色表现对中国国家队来说无疑是一个重大喜讯。目前，国家队在人才储备方面正迫切需要年轻且有实力的新星加入。国足主帅伊万在看到李新翔这场精彩绝伦的比赛之后，很难不被他的天赋和潜力所吸引。鉴于 6 月份即将到来的世预赛，伊万极有可能对李新翔破格提拔，让他有机会在更高级别的赛事中为国家队效力。这位锋线天才神童的出现，无疑为国足注入了新的活力与希望，相信他能在6月的世预赛上真正一战成名，一鸣惊人，带给国足两连胜。\n
"""

print(summarizer(text, 50))

4. 自动更改一段文本表述的语气和风格

In [ ]:
def adjuster(text, des):
    template_string = """
    您的任务是根据一段话来更改一段话的语言风格或语气。\
    请对三个反引号之间的文本进行语言风格及语气变更操作，将文本转变为{style}的语言风格或语气。\
    文本内容: ```{text}```\
     """

    prompt_template = ChatPromptTemplate.from_template(template_string)

    style = f"""正式普通话 \
    用一个{des}的语气\
    """

    messages = prompt_template.format_messages(style=style, text=text)

    return get_completion_from_messages(messages)

text = "小老弟，我小杨，上回你说咱部门要采购的显示器是多少寸来着？"
print(adjuster(text, "正式，客气"))

除了定义单个功能的工具外，我们还可以定义多个工具来串行使用，从而帮助我们复杂任务。
下述例子中，我们通过语言判别器与翻译器实现了，对一段文本的进行语言识别，并翻译成目标语言的功能。

In [ ]:
def lan_discriminator(text):
    template_string = """
    您的任务是根据一段话来判断该段话使用的是什么语种。
    请对三个反引号之间的文本进行语言判断操作，直接输出语种，如法语，无需输出标点符号: 。
    文本内容: ```{text}```
     """
    template = ChatPromptTemplate.from_template(template_string)
    messages = template.format_messages(text=text)
    return get_completion_from_messages(messages=messages)

def translator(text, target_lan):
    lan = lan_discriminator(text)
    template_string = """
    您的任务是将一段{lan}的表述翻译为{target_lan}。\
    请对三个反引号之间的文本进行翻译操作，请直接输出内容，不要带上三个反引号。\
    文本内容: ```{text}```\
     """
    template = ChatPromptTemplate.from_template(template_string)
    messages = template.format_messages(lan=lan, target_lan=target_lan, text=text)
    return get_completion_from_messages(messages=messages)

text = "Il mio mouse non funziona"
print(translator(text, "英文"))

提示词工程正在发展成为一门专业的技能，随着大模型能力的不断提升，精心设计的提示词将成为释放AI潜力的关键。掌握这门艺术，开发者能够在各种应用场景中实现更精准、更高效、更智能的人机交互体验。

提示词工程正在发展成为一门专业的技能，随着大模型能力的不断提升，精心设计的提示词将成为释放AI潜力的关键。掌握这门艺术，开发者能够在各种应用场景中实现更精准、更高效、更智能的人机交互体验。